# Multimodal RAG Educational Assistant — Pipeline Demo

In [33]:
import time
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

print("=" * 60)
print("Multimodal RAG Educational Assistant — Pipeline Demo")
print("=" * 60)

from backend.pipeline.loader import load_file
print("Imported loader.py        — Stage 1: Input Loading (PDF / image / audio)")

from backend.pipeline.preprocessor import preprocess
print("Imported preprocessor.py  — Stage 2: Preprocessing (clean + chunk text)")

from backend.pipeline.embedder import embed
print("Imported embedder.py      — Stage 3: Embedding (all-MiniLM-L6-v2)")

from backend.pipeline.vector_store import build_and_save, load
print("Imported vector_store.py  — Stage 4: Vector Store (FAISS IndexFlatL2)")

from backend.pipeline.retriever import retrieve
print("Imported retriever.py     — Stage 5: Retrieval (top-k similarity search)")

from backend.pipeline.generator import generate
print("Imported generator.py     — Stage 6: Generation (google/flan-t5-large)")

print("=" * 60)
print("All pipeline modules imported successfully")
print("=" * 60)


Multimodal RAG Educational Assistant — Pipeline Demo
Imported loader.py        — Stage 1: Input Loading (PDF / image / audio)
Imported preprocessor.py  — Stage 2: Preprocessing (clean + chunk text)
Imported embedder.py      — Stage 3: Embedding (all-MiniLM-L6-v2)
Imported vector_store.py  — Stage 4: Vector Store (FAISS IndexFlatL2)
Imported retriever.py     — Stage 5: Retrieval (top-k similarity search)
Imported generator.py     — Stage 6: Generation (google/flan-t5-large)
All pipeline modules imported successfully


## PDF Demonstration — Stage 1: Input Loading

In [34]:
print("=" * 60)
print("STAGE 1: INPUT LOADING — PDF")
print("=" * 60)

start = time.time()
pdf_text = load_file("../data/Prototype/sample_document.pdf")
pdf_load_time = time.time() - start

print("Extracted text:")
print(pdf_text)
print("-" * 60)
print(f"Extraction time: {pdf_load_time:.2f}s")


[loader] Auto-detected '.pdf' → input_type='pdf'
[loader] Loading PDF → ../data/Prototype/sample_document.pdf
[loader] PDF loaded — 1 pages, 306 characters


STAGE 1: INPUT LOADING — PDF
Extracted text:
Machine learning (ML) is a subset of artificial intelligence (AI) that teaches computers 
to learn from data and identify patterns without being explicitly programmed for every 
specific task. Instead of following hard-coded rules, ML models improve their accuracy over 
time by analyzing large datasets  

------------------------------------------------------------
Extraction time: 0.00s


## PDF Demonstration — Stage 2: Preprocessing

In [35]:
print("=" * 60)
print("STAGE 2: PREPROCESSING — PDF")
print("=" * 60)

pdf_chunks = preprocess(pdf_text)

print(f"Number of chunks: {len(pdf_chunks)}")
print("-" * 60)
for i, chunk in enumerate(pdf_chunks):
    print(f"Chunk [{i}]:")
    print(chunk)
    print("-" * 60)


STAGE 2: PREPROCESSING — PDF
Number of chunks: 1
------------------------------------------------------------
Chunk [0]:
machine learning ( ml ) is a subset of artificial intelligence ( ai ) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml models improve their accuracy over time by analyzing large datasets
------------------------------------------------------------


## PDF Demonstration — Stage 3: Embedding

In [36]:
print("=" * 60)
print("STAGE 3: EMBEDDING — PDF")
print("=" * 60)

pdf_embeddings = embed(pdf_chunks)

print(f"Embedding shape: {pdf_embeddings.shape}  (n_chunks x 384)")


STAGE 3: EMBEDDING — PDF
Embedding shape: (1, 384)  (n_chunks x 384)


## PDF Demonstration — Stage 4: Vector Store

In [37]:
print("=" * 60)
print("STAGE 4: VECTOR STORE — PDF")
print("=" * 60)

build_and_save(pdf_embeddings, pdf_chunks)
pdf_index, pdf_stored_chunks = load()

print(f"Total vectors stored in FAISS index: {pdf_index.ntotal}")


STAGE 4: VECTOR STORE — PDF
Total vectors stored in FAISS index: 1


## PDF Demonstration — Stage 5: Retrieval

In [38]:
print("=" * 60)
print("STAGE 5: RETRIEVAL — PDF")
print("=" * 60)

question = "What is machine learning, and how do ML models improve over time?"
pdf_retrieved = retrieve(question, pdf_index, pdf_stored_chunks, k=3)

print(f"Question: {question}")
print("-" * 60)
print("Retrieved chunks:")
for i, chunk in enumerate(pdf_retrieved, start=1):
    print(f"[{i}] {chunk}")
    print("-" * 60)


STAGE 5: RETRIEVAL — PDF
Question: What is machine learning, and how do ML models improve over time?
------------------------------------------------------------
Retrieved chunks:
[1] machine learning ( ml ) is a subset of artificial intelligence ( ai ) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml models improve their accuracy over time by analyzing large datasets
------------------------------------------------------------


## PDF Demonstration — Stage 6: Generation

In [39]:
print("=" * 60)
print("STAGE 6: GENERATION — PDF")
print("=" * 60)

start = time.time()
pdf_answer = generate(question, pdf_retrieved)
pdf_gen_time = time.time() - start

print(f"Generated answer: {pdf_answer}")
print("-" * 60)
print(f"Generation time: {pdf_gen_time:.2f}s")


STAGE 6: GENERATION — PDF
Generated answer: a subset of artificial intelligence ( ai ) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
------------------------------------------------------------
Generation time: 2.74s


## PDF Demonstration — Summary

In [40]:
print("=" * 60)
print("SUMMARY — PDF")
print("=" * 60)
print(f"Question         : {question}")
print(f"Generated Answer : {pdf_answer}")
print(f"Source Modality  : PDF")
print("=" * 60)


SUMMARY — PDF
Question         : What is machine learning, and how do ML models improve over time?
Generated Answer : a subset of artificial intelligence ( ai ) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
Source Modality  : PDF


## Image Demonstration — Stage 1: Input Loading

In [56]:
print("=" * 60)
print("STAGE 1: INPUT LOADING — IMAGE")
print("=" * 60)

start = time.time()
img_text = load_file("../data/Prototype/sample_image.png")
img_load_time = time.time() - start

print("Extracted text:")
print(img_text)
print("-" * 60)
print(f"Extraction time: {img_load_time:.2f}s")

# Detect which image extraction path was used:
# Qwen2-VL output looks like structured markdown/prose (transcription + chart/figure description).
# The EasyOCR+BLIP fallback output is recognizable because it always starts with the literal
# prefix "Extracted text:" (see backend/pipeline/loader.py::_load_image_easyocr_blip).
if img_text.startswith("Extracted text:"):
    print("Extraction path: EasyOCR + BLIP fallback was used (Qwen2-VL failed to load/run)")
else:
    print("Extraction path: Qwen2-VL-2B-Instruct (primary extractor) was used")


[loader] Auto-detected '.png' → input_type='image'
[loader] Processing image → ../data/Prototype/sample_image.png


STAGE 1: INPUT LOADING — IMAGE
Extracted text:
The text describes machine learning (ML) as a subset of artificial intelligence (AI) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. Instead of following hard-coded rules, ML models improve their accuracy over time by analyzing large datasets. The text also mentions IBM as a contributor to the field of ML.
------------------------------------------------------------
Extraction time: 5.27s
Extraction path: Qwen2-VL-2B-Instruct (primary extractor) was used


## Image Demonstration — Stage 2: Preprocessing

In [42]:
print("=" * 60)
print("STAGE 2: PREPROCESSING — IMAGE")
print("=" * 60)

img_chunks = preprocess(img_text)

print(f"Number of chunks: {len(img_chunks)}")
print("-" * 60)
for i, chunk in enumerate(img_chunks):
    print(f"Chunk [{i}]:")
    print(chunk)
    print("-" * 60)


STAGE 2: PREPROCESSING — IMAGE
Number of chunks: 1
------------------------------------------------------------
Chunk [0]:
the text in the document is a description of machine learning ( ml ) as a subset of artificial intelligence ( ai ). it explains that ml teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml models improve their accuracy over time by analyzing large datasets. there are no charts, tables, or figures present in the text. the text provides a straightforward explanation of what ml is and how it works.
------------------------------------------------------------


## Image Demonstration — Stage 3: Embedding

In [43]:
print("=" * 60)
print("STAGE 3: EMBEDDING — IMAGE")
print("=" * 60)

img_embeddings = embed(img_chunks)

print(f"Embedding shape: {img_embeddings.shape}  (n_chunks x 384)")


STAGE 3: EMBEDDING — IMAGE
Embedding shape: (1, 384)  (n_chunks x 384)


## Image Demonstration — Stage 4: Vector Store

In [44]:
print("=" * 60)
print("STAGE 4: VECTOR STORE — IMAGE")
print("=" * 60)

build_and_save(img_embeddings, img_chunks)
img_index, img_stored_chunks = load()

print(f"Total vectors stored in FAISS index: {img_index.ntotal}")


STAGE 4: VECTOR STORE — IMAGE
Total vectors stored in FAISS index: 1


## Image Demonstration — Stage 5: Retrieval

In [45]:
print("=" * 60)
print("STAGE 5: RETRIEVAL — IMAGE")
print("=" * 60)

question = "What is machine learning, and how do ML models improve over time?"
img_retrieved = retrieve(question, img_index, img_stored_chunks, k=3)

print(f"Question: {question}")
print("-" * 60)
print("Retrieved chunks:")
for i, chunk in enumerate(img_retrieved, start=1):
    print(f"[{i}] {chunk}")
    print("-" * 60)


STAGE 5: RETRIEVAL — IMAGE
Question: What is machine learning, and how do ML models improve over time?
------------------------------------------------------------
Retrieved chunks:
[1] the text in the document is a description of machine learning ( ml ) as a subset of artificial intelligence ( ai ). it explains that ml teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml models improve their accuracy over time by analyzing large datasets. there are no charts, tables, or figures present in the text. the text provides a straightforward explanation of what ml is and how it works.
------------------------------------------------------------


## Image Demonstration — Stage 6: Generation

In [46]:
print("=" * 60)
print("STAGE 6: GENERATION — IMAGE")
print("=" * 60)

start = time.time()
img_answer = generate(question, img_retrieved)
img_gen_time = time.time() - start

print(f"Generated answer: {img_answer}")
print("-" * 60)
print(f"Generation time: {img_gen_time:.2f}s")


STAGE 6: GENERATION — IMAGE
Generated answer: teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
------------------------------------------------------------
Generation time: 1.61s


## Image Demonstration — Summary

In [47]:
print("=" * 60)
print("SUMMARY — IMAGE")
print("=" * 60)
print(f"Question         : {question}")
print(f"Generated Answer : {img_answer}")
print(f"Source Modality  : Image")
print("=" * 60)


SUMMARY — IMAGE
Question         : What is machine learning, and how do ML models improve over time?
Generated Answer : teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
Source Modality  : Image


## Audio Demonstration — Stage 1: Input Loading

In [48]:
print("=" * 60)
print("STAGE 1: INPUT LOADING — AUDIO")
print("=" * 60)

start = time.time()
aud_text = load_file("../data/Prototype/sample_audio.m4a")
aud_load_time = time.time() - start

print("Extracted text:")
print(aud_text)
print("-" * 60)
print(f"Extraction time: {aud_load_time:.2f}s")


[loader] Auto-detected '.m4a' → input_type='audio'
[loader] Transcribing audio → ../data/Prototype/sample_audio.m4a  (model: base)


STAGE 1: INPUT LOADING — AUDIO


[loader] Transcription complete — 313 characters


Extracted text:
 Machine Learning is a subset of artificial intelligence that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. Instead of following hard-coded rules, ML or machine learning, models improve their accuracy over time by analyzing large datasets.
------------------------------------------------------------
Extraction time: 1.37s


## Audio Demonstration — Stage 2: Preprocessing

In [49]:
print("=" * 60)
print("STAGE 2: PREPROCESSING — AUDIO")
print("=" * 60)

aud_chunks = preprocess(aud_text)

print(f"Number of chunks: {len(aud_chunks)}")
print("-" * 60)
for i, chunk in enumerate(aud_chunks):
    print(f"Chunk [{i}]:")
    print(chunk)
    print("-" * 60)


STAGE 2: PREPROCESSING — AUDIO
Number of chunks: 1
------------------------------------------------------------
Chunk [0]:
machine learning is a subset of artificial intelligence that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml or machine learning, models improve their accuracy over time by analyzing large datasets.
------------------------------------------------------------


## Audio Demonstration — Stage 3: Embedding

In [50]:
print("=" * 60)
print("STAGE 3: EMBEDDING — AUDIO")
print("=" * 60)

aud_embeddings = embed(aud_chunks)

print(f"Embedding shape: {aud_embeddings.shape}  (n_chunks x 384)")


STAGE 3: EMBEDDING — AUDIO
Embedding shape: (1, 384)  (n_chunks x 384)


## Audio Demonstration — Stage 4: Vector Store

In [51]:
print("=" * 60)
print("STAGE 4: VECTOR STORE — AUDIO")
print("=" * 60)

build_and_save(aud_embeddings, aud_chunks)
aud_index, aud_stored_chunks = load()

print(f"Total vectors stored in FAISS index: {aud_index.ntotal}")


STAGE 4: VECTOR STORE — AUDIO
Total vectors stored in FAISS index: 1


## Audio Demonstration — Stage 5: Retrieval

In [52]:
print("=" * 60)
print("STAGE 5: RETRIEVAL — AUDIO")
print("=" * 60)

question = "What is machine learning, and how do ML models improve over time?"
aud_retrieved = retrieve(question, aud_index, aud_stored_chunks, k=3)

print(f"Question: {question}")
print("-" * 60)
print("Retrieved chunks:")
for i, chunk in enumerate(aud_retrieved, start=1):
    print(f"[{i}] {chunk}")
    print("-" * 60)


STAGE 5: RETRIEVAL — AUDIO
Question: What is machine learning, and how do ML models improve over time?
------------------------------------------------------------
Retrieved chunks:
[1] machine learning is a subset of artificial intelligence that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task. instead of following hard - coded rules, ml or machine learning, models improve their accuracy over time by analyzing large datasets.
------------------------------------------------------------


## Audio Demonstration — Stage 6: Generation

In [53]:
print("=" * 60)
print("STAGE 6: GENERATION — AUDIO")
print("=" * 60)

start = time.time()
aud_answer = generate(question, aud_retrieved)
aud_gen_time = time.time() - start

print(f"Generated answer: {aud_answer}")
print("-" * 60)
print(f"Generation time: {aud_gen_time:.2f}s")


STAGE 6: GENERATION — AUDIO
Generated answer: a subset of artificial intelligence that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
------------------------------------------------------------
Generation time: 2.22s


## Audio Demonstration — Summary

In [54]:
print("=" * 60)
print("SUMMARY — AUDIO")
print("=" * 60)
print(f"Question         : {question}")
print(f"Generated Answer : {aud_answer}")
print(f"Source Modality  : Audio")
print("=" * 60)


SUMMARY — AUDIO
Question         : What is machine learning, and how do ML models improve over time?
Generated Answer : a subset of artificial intelligence that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task
Source Modality  : Audio


## Final Cross-Modality Comparison

In [55]:
print("=" * 60)
print("FINAL CROSS-MODALITY COMPARISON")
print("=" * 60)

key_phrases = ["data", "artificial intelligence"]

def contains_key_phrase(answer, phrases):
    lowered = answer.lower()
    return any(phrase in lowered for phrase in phrases)

modalities = [
    ("PDF",   pdf_text,  pdf_answer),
    ("Image", img_text,  img_answer),
    ("Audio", aud_text,  aud_answer),
]

flags = [contains_key_phrase(answer, key_phrases) for _, _, answer in modalities]
consistent = all(flags)

print(f"{'Modality':<8} | {'Extracted (first 100 chars)':<60} | {'Generated Answer':<40} | Answer Consistent?")
print("-" * 130)
for (name, text, answer), flag in zip(modalities, flags):
    preview = text[:100].replace(chr(10), ' ')
    print(f"{name:<8} | {preview:<60} | {answer:<40} | {flag}")

print("-" * 130)
print(f"All three answers share a common key phrase ({key_phrases}): {consistent}")
print("=" * 60)


FINAL CROSS-MODALITY COMPARISON
Modality | Extracted (first 100 chars)                                  | Generated Answer                         | Answer Consistent?
----------------------------------------------------------------------------------------------------------------------------------
PDF      | Machine learning (ML) is a subset of artificial intelligence (AI) that teaches computers  to learn f | a subset of artificial intelligence ( ai ) that teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task | True
Image    | The text in the document is a description of machine learning (ML) as a subset of artificial intelli | teaches computers to learn from data and identify patterns without being explicitly programmed for every specific task | True
Audio    |  Machine Learning is a subset of artificial intelligence that teaches computers to learn from data a | a subset of artificial intelligence that teaches computers 